In [7]:
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx spacy datasets langchain-community llama-index


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB

In [87]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = "/content/hackernoon_subset.csv"
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40


In [9]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "/content/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 5_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise


Đang kết nối luồng dữ liệu (streaming)...


README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Đang ghi dữ liệu vào: /content/hackernoon_subset.csv


Đang tải (MB):   0%|          | 0/300 [00:00<?, ?MB/s]


[DỪNG] Đã đạt giới hạn số dòng: 5,000 dòng (Dung lượng: 2.92 MB)
✅ Hoàn thành: /content/hackernoon_subset.csv
   Rows: 5,000
   Size: 2.92 MB


In [10]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()


✅ Neo4j connected.
✅ Schema ready.


In [11]:
#@title 1.5
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    # Added 'description' to candidates to support HackerNoon dataset
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)

# === Challenge A (bonus) — Near-Duplicate Detection via MinHash/LSH ===
# Chạy SAU standardize_news() (đã exact-dedup bằng SHA1) và TRƯỚC build_chunks().
# Mục tiêu: bắt các bài near-duplicate (repost/syndicate/sửa nhẹ) mà exact-hash bỏ sót,
# KHÔNG dùng pairwise cosine O(N^2) — dùng MinHash signature + LSH bucket để chỉ so
# các cặp candidate có khả năng trùng cao.

!pip -q install datasketch

from datasketch import MinHash, MinHashLSH

# ---- Tham số (đưa vào report) ----
SHINGLE_N = 5          # kích thước word n-gram dùng làm shingle
NUM_PERM = 128         # số permutation cho MinHash (đánh đổi độ chính xác vs tốc độ/RAM)
LSH_THRESHOLD = 0.80   # ngưỡng Jaccard similarity ước lượng để coi là near-duplicate
AUDIT_SAMPLE_SIZE = 15 # số cặp lấy mẫu để audit thủ công + ước lượng false positive rate

def make_shingles(text, n=SHINGLE_N):
    words = norm_space(text).lower().split()
    if len(words) < n:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

def make_minhash(text, num_perm=NUM_PERM):
    mh = MinHash(num_perm=num_perm)
    for sh in make_shingles(text):
        mh.update(sh.encode("utf-8"))
    return mh

def jaccard_exact(a_text, b_text, n=SHINGLE_N):
    sa, sb = make_shingles(a_text, n), make_shingles(b_text, n)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def find_near_duplicates(df, text_col="text", id_col="article_id",
                          threshold=LSH_THRESHOLD, num_perm=NUM_PERM):
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    minhashes = {}
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="MinHash build"):
        rid = getattr(row, id_col)
        text = f"{getattr(row, 'title', '')} {getattr(row, text_col)}"
        mh = make_minhash(text, num_perm=num_perm)
        minhashes[rid] = mh
        lsh.insert(rid, mh)

    parent = {rid: rid for rid in minhashes}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry

    pair_rows, seen_pairs = [], set()
    id_to_text = dict(zip(df[id_col], df[text_col]))
    for rid, mh in tqdm(minhashes.items(), desc="LSH query"):
        for cand in lsh.query(mh):
            if cand == rid:
                continue
            key = tuple(sorted((rid, cand)))
            if key in seen_pairs:
                continue
            seen_pairs.add(key)
            sim = jaccard_exact(id_to_text[key[0]], id_to_text[key[1]])
            pair_rows.append({"id_a": key[0], "id_b": key[1], "jaccard_est": sim})
            union(rid, cand)

    pairs_df = pd.DataFrame(pair_rows).sort_values("jaccard_est", ascending=False).reset_index(drop=True)

    clusters = {}
    for rid in minhashes:
        clusters.setdefault(find(rid), []).append(rid)
    keep_ids = set()
    for members in clusters.values():
        keep_ids.add(members[0] if len(members) == 1
                      else max(members, key=lambda i: len(id_to_text.get(i, ""))))
    return keep_ids, pairs_df

# ---- Chạy trên news_df ----
before_n = len(news_df)
keep_ids, near_dup_pairs_df = find_near_duplicates(news_df, threshold=LSH_THRESHOLD)
news_df_before_near_dedup = news_df.copy()
news_df = news_df[news_df["article_id"].isin(keep_ids)].reset_index(drop=True)

print(f"Near-dedup (MinHash/LSH, threshold={LSH_THRESHOLD}): {before_n:,} -> {len(news_df):,}")
print(f"Số cặp near-duplicate tìm được: {len(near_dup_pairs_df):,}")
display(near_dup_pairs_df.head(10))

# ---- Audit sample để ước lượng false positive rate ----
if len(near_dup_pairs_df) > 0:
    audit_sample = near_dup_pairs_df.sample(min(AUDIT_SAMPLE_SIZE, len(near_dup_pairs_df)), random_state=SEED).copy()
    title_map = dict(zip(news_df_before_near_dedup["article_id"], news_df_before_near_dedup["title"]))
    text_map = dict(zip(news_df_before_near_dedup["article_id"], news_df_before_near_dedup["text"].str.slice(0, 200)))
    audit_sample["title_a"] = audit_sample["id_a"].map(title_map)
    audit_sample["title_b"] = audit_sample["id_b"].map(title_map)
    audit_sample["text_a_preview"] = audit_sample["id_a"].map(text_map)
    audit_sample["text_b_preview"] = audit_sample["id_b"].map(text_map)
    display(audit_sample[["id_a","id_b","jaccard_est","title_a","title_b","text_a_preview","text_b_preview"]])
    audit_sample.to_csv("near_dup_audit_sample.csv", index=False)
    print("Đã lưu near_dup_audit_sample.csv — audit thủ công, tính FP_rate = (cặp gắn sai)/(tổng cặp audit).")
else:
    print("Không tìm thấy cặp near-duplicate nào ở threshold hiện tại.")
chunks_df = build_chunks(news_df)
display(chunks_df.head())


Exact dedup: 2,675 -> 2,105
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 3.7 MB/s eta 0:00:00


MinHash build:   0%|          | 0/1500 [00:00<?, ?it/s]

LSH query:   0%|          | 0/1500 [00:00<?, ?it/s]

Near-dedup (MinHash/LSH, threshold=0.8): 1,500 -> 1,496
Số cặp near-duplicate tìm được: 4


,id_a,id_b,jaccard_est
0,d2f10b6eb420a221b002,e41e603c9b2b467c7a45,1.000000
1,20900bb77008b0464c81,e6090305820383267481,1.000000
2,0366643faf4dd2a89fa4,ef887c1d851c608d9f68,0.884615
3,99e4dff966ca14987a42,e1dd933ce175e571f1fc,0.793103


,id_a,id_b,jaccard_est,title_a,title_b,text_a_preview,text_b_preview
1,20900bb77008b0464c81,e6090305820383267481,1.000000,411 is going out of service for millions of Americans,411 phone number is going out of service for millions of Americans,AT&T customers with digital landlines won''t be able to dial 411 or 0 to reach an operator or get directory assistan...,AT&T customers with digital landlines won''t be able to dial 411 or 0 to reach an operator or get directory assistan...
3,99e4dff966ca14987a42,e1dd933ce175e571f1fc,0.793103,Knicks rally to beat Pacers 109-106 for 7th straight victory,Knicks rally to beat Pacers 109-106 for 7th straight victory,Jalen Brunson scored 30 points Julius Randle made six free throws in the final minute and the New York Knicks beat t...,Jalen Brunson scored 30 points Julius Randle made six free throws in the final minute and the New York Knicks beat t...
0,d2f10b6eb420a221b002,e41e603c9b2b467c7a45,1.000000,CereCore® expands healthcare technology services into the UK,CereCore expands healthcare technology services into the UK,NASHVILLE Tenn. Dec. 8 2022 /PRNewswire/ -- CereCore today announced they are expanding their healthcare information...,NASHVILLE Tenn. Dec. 8 2022 /PRNewswire/ -- CereCore today announced they are expanding their healthcare information...
2,0366643faf4dd2a89fa4,ef887c1d851c608d9f68,0.884615,US Announces Criminal Cases Involving Flow of Technology Information to Russia China and Iran,US announces criminal cases involving flow of technology information to Russia China and Iran,WASHINGTON — The Justice Department announced a series of criminal cases Tuesday tracing the illegal flow of sensiti...,WASHINGTON (AP) — The Justice Department announced a series of criminal cases Tuesday tracing the illegal flow of se...


Đã lưu near_dup_audit_sample.csv — audit thủ công, tính FP_rate = (cặp gắn sai)/(tổng cặp audit).


Chunking:   0%|          | 0/1496 [00:00<?, ?it/s]

,chunk_id,article_id,title,published_date,text
0,1a05beb7aa3071be6fd7::c0000,1a05beb7aa3071be6fd7,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,8e922bc62b578e73e815::c0000,8e922bc62b578e73e815,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
2,4bd7afdba71243b0dbcd::c0000,4bd7afdba71243b0dbcd,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
3,6633c15d86f5e81f47b8::c0000,6633c15d86f5e81f47b8,5 Kubernetes technology vendors hot right now,2023-02-28,Kubernetes is a technology that has created a whole new ecosystem around itself and it is now a key plank in the Dev...
4,4ce72a4490a6a618e2d5::c0000,4ce72a4490a6a618e2d5,Bachelor of Science in Health Information Management,2023-08-16,Health information management (HIM) is a diverse yet evolving field that incorporates medicine management finance in...


In [90]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None


def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])


# ---- Parse "try again in Xm Ys" / "try again in Xs" từ lỗi 429 của Groq ----
RETRY_AFTER_RE = re.compile(r"try again in (\d+)m([\d.]+)s|try again in ([\d.]+)s", re.IGNORECASE)


def _parse_retry_after_seconds(error_str, cap=90):
    """
    cap: nếu thời gian Groq yêu cầu chờ > cap giây, coi như quota ngày gần cạn,
    raise ngay thay vì sleep lâu vô ích.
    """
    m = RETRY_AFTER_RE.search(str(error_str))
    if not m:
        return None
    if m.group(1) is not None:
        return int(m.group(1)) * 60 + float(m.group(2))
    return float(m.group(3))


def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {"model": model, "messages": messages, "temperature": 0.0}
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            if "gpt-oss" in model:
                # Reasoning content của GPT-OSS models mặc định include_reasoning=True,
                # đôi khi lẫn vào 'content' và làm hỏng cấu trúc JSON lồng nhau.
                kwargs["include_reasoning"] = False

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            err_str = str(e)
            if "429" in err_str or "rate_limit" in err_str.lower():
                wait_s = _parse_retry_after_seconds(err_str)
                if wait_s is not None and wait_s <= 90:
                    time.sleep(wait_s + 1)
                    continue
                if wait_s is not None:
                    raise RuntimeError(f"RATE_LIMIT_LIKELY_EXHAUSTED: {err_str}") from e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2 ** attempt + random.random()))
    raise RuntimeError(last)


def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage


test_result, usage = groq_json(
    system="Bạn là extractor. Trả về JSON.",
    user='Trích xuất tên công ty và loại thực thể từ câu: "Apple ra mắt iPhone mới tại California."'
)
print(test_result)
print(usage)

print(bool(GROQ_API_KEY), bool(GROQ_MODEL))

{'company': 'Apple', 'entity_type': 'Company'}
{'prompt_tokens': 134, 'completion_tokens': 193, 'total_tokens': 327}
True True


In [13]:
#@title 1.7 — Coreference resolution theo batch (+ fact-preservation validation)

COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]
    prompt = f"""
Resolve coreferences.
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}
INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}
    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

# ---- Fact-preservation validation ----
# Failure mode: false coreference -> false edge. Regex không chặn được 100%
# nhưng phát hiện lệch để đưa vào audit queue.

NUM_RE = re.compile(r"\b\d[\d,.]*\b")
DATE_RE = re.compile(
    r"\b(?:\d{4}-\d{2}-\d{2}"
    r"|\d{1,2}/\d{1,2}/\d{2,4}"
    r"|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2}(?:,\s*\d{4})?)\b",
    re.IGNORECASE,
)
TICKER_RE = re.compile(r"\b[A-Z]{2,5}\b")

def extract_facts(text):
    return {
        "numbers": set(NUM_RE.findall(text or "")),
        "dates": set(DATE_RE.findall(text or "")),
        "tickers": set(TICKER_RE.findall(text or "")),
    }

def validate_fact_preservation(orig_text, resolved_text):
    a, b = extract_facts(orig_text), extract_facts(resolved_text)
    flags = []
    for key in ("numbers", "dates", "tickers"):
        missing = a[key] - b[key]
        added = b[key] - a[key]
        if missing:
            flags.append(f"{key}_missing:{sorted(missing)}")
        if added:
            flags.append(f"{key}_added:{sorted(added)}")
    return flags

# ---- Kích hoạt chạy thật ----
extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

extraction_source["coref_fact_flags"] = extraction_source.apply(
    lambda r: validate_fact_preservation(r["text"], r["resolved_text"]), axis=1
)
extraction_source["coref_has_unresolved"] = extraction_source["unresolved_mentions"].map(
    lambda x: bool(x) and x != ["COREF_BATCH_FAILED"]
)
extraction_source["coref_batch_failed"] = extraction_source["unresolved_mentions"].map(
    lambda x: x == ["COREF_BATCH_FAILED"]
)

n_total = len(extraction_source)
n_failed = extraction_source["coref_batch_failed"].sum()
n_unresolved = extraction_source["coref_has_unresolved"].sum()
n_fact_flagged = (extraction_source["coref_fact_flags"].map(len) > 0).sum()

print(f"Coref chạy trên {n_total:,} chunks")
print(f"  - Batch fail (fallback về text gốc): {n_failed:,}")
print(f"  - Có unresolved_mentions (ambiguity, giữ nguyên): {n_unresolved:,}")
print(f"  - Bị flag lệch fact (số/ngày/ticker) — cần audit: {n_fact_flagged:,}")

if n_fact_flagged > 0:
    flagged = extraction_source[extraction_source["coref_fact_flags"].map(len) > 0]
    display(flagged[["chunk_id", "text", "resolved_text", "coref_fact_flags"]].head(10))
    flagged.to_csv("coref_fact_audit.csv", index=False)
    print("Đã lưu coref_fact_audit.csv để audit.")

display(extraction_source.head())

Coref:   0%|          | 0/80 [00:00<?, ?it/s]

Coref chạy trên 400 chunks
  - Batch fail (fallback về text gốc): 0
  - Có unresolved_mentions (ambiguity, giữ nguyên): 61
  - Bị flag lệch fact (số/ngày/ticker) — cần audit: 2


,chunk_id,text,resolved_text,coref_fact_flags
70,d7780588f2f2dfd15ca1::c0000,Freshworks Inc. (NASDAQ: FRSH) today announced the launch of its AI-powered Customer Service Suite which brings toge...,Freshworks Inc. today announced the launch of Freshworks Inc.'s AI-powered Customer Service Suite which brings toget...,[tickers_missing:['FRSH']]
169,8dc0de1ac5eca8c4bb72::c0000,The company’s multiple products were then centralised in a single Akeneo product information ... services company ha...,The company’s multiple products were then centralised in a single Akeneo product information ... services company ha...,[tickers_missing:['ML']]


Đã lưu coref_fact_audit.csv để audit.


,chunk_id,article_id,title,published_date,text,resolved_text,unresolved_mentions,coref_fact_flags,coref_has_unresolved,coref_batch_failed
0,1a05beb7aa3071be6fd7::c0000,1a05beb7aa3071be6fd7,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...,[],[],False,False
1,8e922bc62b578e73e815::c0000,8e922bc62b578e73e815,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...,[It],[],True,False
2,4bd7afdba71243b0dbcd::c0000,4bd7afdba71243b0dbcd,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...,[],[],False,False
3,6633c15d86f5e81f47b8::c0000,6633c15d86f5e81f47b8,5 Kubernetes technology vendors hot right now,2023-02-28,Kubernetes is a technology that has created a whole new ecosystem around itself and it is now a key plank in the Dev...,Kubernetes is a technology that has created a whole new ecosystem around itself and Kubernetes is now a key plank in...,[],[],False,False
4,4ce72a4490a6a618e2d5::c0000,4ce72a4490a6a618e2d5,Bachelor of Science in Health Information Management,2023-08-16,Health information management (HIM) is a diverse yet evolving field that incorporates medicine management finance in...,Health information management (HIM) is a diverse yet evolving field that incorporates medicine management finance in...,[],[],False,False


In [14]:
# ---- Safety net: revert các chunk bị flag lệch fact về text gốc ----
# Lý do: false coreference -> false edge. Với 2/400 chunk bị flag (0.5%),
# cách an toàn nhất là không tin resolved_text của LLM cho các chunk này.
mask_flagged = extraction_source["coref_fact_flags"].map(len) > 0
n_reverted = mask_flagged.sum()

extraction_source.loc[mask_flagged, "resolved_text"] = extraction_source.loc[mask_flagged, "text"]
extraction_source.loc[mask_flagged, "unresolved_mentions"] = extraction_source.loc[mask_flagged, "unresolved_mentions"].apply(
    lambda x: (x if isinstance(x, list) else []) + ["REVERTED_DUE_TO_FACT_FLAG"]
)

print(f"Đã revert {n_reverted} chunk về text gốc do bị flag lệch fact (audit: coref_fact_audit.csv)")

Đã revert 2 chunk về text gốc do bị flag lệch fact (audit: coref_fact_audit.csv)


In [ ]:
flagged_ids = ["76879c031f77eb7392eb::c0000", "70175d5248e739d351ec::c0000"]

# 1) Có mặt trong extraction_source (đầu vào Section 2) không?
in_source = extraction_source[extraction_source["chunk_id"].isin(flagged_ids)]
print("Trong extraction_source:", len(in_source), "/", len(flagged_ids))
print(in_source[["chunk_id"]])

# 2) Có sinh ra triples nào trong raw_triples_df không? (cột là source_chunk_id, không phải chunk_id)
in_triples = raw_triples_df[raw_triples_df["source_chunk_id"].isin(flagged_ids)]
print("\nCó triples trong raw_triples_df:", len(in_triples))

# 3) Có bị guardrail loại (dropped) không?
in_dropped = dropped_relations_df[dropped_relations_df["chunk_id"].isin(flagged_ids)]
print("Có trong dropped_relations_df:", len(in_dropped))
print(in_dropped)

# 4) Batch (start) chứa 2 chunk này nằm ở đâu trong extraction_source, để tra xem batch đó
#    có từng nằm trong extraction_errors_df lịch sử không (dù giờ đã resume xong)
idx_positions = [extraction_source.index.get_loc(i) for i in in_source.index]
print("\nVị trí (index) trong extraction_source:", idx_positions)
print("Batch (start, batch_size=4) tương ứng:", [ (p // 4) * 4 for p in idx_positions ])

Trong extraction_source: 2 / 2
                        chunk_id
216  76879c031f77eb7392eb::c0000
261  70175d5248e739d351ec::c0000

Có triples trong raw_triples_df: 1
Có trong dropped_relations_df: 0
Empty DataFrame
Columns: [chunk_id, source_raw, relation, target_raw, reason]
Index: []

Vị trí (index) trong extraction_source: [216, 261]
Batch (start, batch_size=4) tương ứng: [216, 260]


In [39]:
#@title 2.1 NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()


def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]
    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}
INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)


def run_extraction_on_slice(batch_df, meta, min_confidence=0.0, start_label=0):
    """
    Chạy extraction cho ĐÚNG 1 batch đã cho (không tự chia nhỏ thêm).
    Dùng chung cho cả lần chạy đầu (qua run_extraction) lẫn lúc resume.
    start_label: nhãn batch (vị trí gốc trong source_df) để ghi vào error/dropped,
    dùng để resume chính xác batch đó về sau.
    """
    triples, errors, dropped = [], [], []
    try:
        obj, _ = extract_batch(batch_df)
    except Exception as e:
        errors.append({"start": start_label, "error": str(e), "stage": "api_call"})
        return pd.DataFrame(triples), pd.DataFrame(errors), pd.DataFrame(dropped)

    try:
        items = obj.get("items", [])
        if not isinstance(items, list):
            raise TypeError(f"'items' không phải list mà là {type(items).__name__}")

        for item in items:
            if not isinstance(item, dict):
                dropped.append({
                    "start": start_label, "chunk_id": None, "source_raw": None,
                    "relation": None, "target_raw": None,
                    "reason": f"malformed_item_type:{type(item).__name__}:{str(item)[:120]}",
                })
                continue

            cid = item.get("chunk_id")
            if cid not in meta:
                continue

            relations = item.get("relations", [])
            if not isinstance(relations, list):
                dropped.append({
                    "start": start_label, "chunk_id": cid, "source_raw": None,
                    "relation": None, "target_raw": None,
                    "reason": f"malformed_relations_type:{type(relations).__name__}",
                })
                continue

            for x in relations:
                if not isinstance(x, dict):
                    dropped.append({
                        "start": start_label, "chunk_id": cid, "source_raw": None,
                        "relation": None, "target_raw": None,
                        "reason": f"malformed_relation_item_type:{type(x).__name__}",
                    })
                    continue

                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                evidence = norm_space(x.get("evidence"))

                reason = None
                if not s or not t:
                    reason = "empty_source_or_target"
                elif st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    reason = "node_type_not_allowed"
                elif rel not in ALLOWED_RELATIONS:
                    reason = "relation_not_allowed"
                elif not evidence:
                    reason = "missing_evidence"
                elif s.lower() == t.lower() and st == tt:
                    reason = "self_loop"

                if reason:
                    dropped.append({"start": start_label, "chunk_id": cid, "source_raw": s,
                                     "relation": rel, "target_raw": t, "reason": reason})
                    continue

                try:
                    conf = float(x.get("confidence"))
                except (TypeError, ValueError):
                    conf = 0.5
                conf = max(0.0, min(1.0, conf))

                if conf < min_confidence:
                    dropped.append({"start": start_label, "chunk_id": cid, "source_raw": s,
                                     "relation": rel, "target_raw": t,
                                     "reason": f"below_min_confidence({conf:.2f})"})
                    continue

                triples.append({
                    "source_raw": s, "source_type": st, "relation": rel,
                    "target_raw": t, "target_type": tt,
                    "source_chunk_id": cid, "published_date": meta[cid] or "",
                    "evidence": evidence, "confidence": conf,
                    "model_used": GROQ_MODEL,
                })

    except Exception as e:
        errors.append({"start": start_label, "error": f"parse_error: {e}", "stage": "parse_response"})

    return pd.DataFrame(triples), pd.DataFrame(errors), pd.DataFrame(dropped)


def run_extraction(source_df, batch_size=4, min_confidence=0.0):
    """Chạy full từ đầu, chia source_df thành các batch theo batch_size."""
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    all_t, all_e, all_d = [], [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start + batch_size]
        t_df, e_df, d_df = run_extraction_on_slice(batch, meta, min_confidence, start_label=start)
        all_t.append(t_df); all_e.append(e_df); all_d.append(d_df)

    triples_df = pd.concat(all_t, ignore_index=True) if all_t else pd.DataFrame()
    errors_df = pd.concat(all_e, ignore_index=True) if all_e else pd.DataFrame()
    dropped_df = pd.concat(all_d, ignore_index=True) if all_d else pd.DataFrame()
    return triples_df, errors_df, dropped_df


def resume_failed_extraction(source_df, prev_triples_df, prev_errors_df, prev_dropped_df,
                              batch_size=4, min_confidence=0.0):
    """
    Chỉ chạy lại các batch từng lỗi api_call/parse_response, hoặc có item bị lỗi
    cấu trúc (malformed_*). Dùng GROQ_MODEL hiện tại (đọc từ biến global, đã set ở cell khác).
    Kết quả mới được gộp với kết quả cũ; batch nào không bị resume thì giữ nguyên.
    """
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()

    starts_to_retry = set(prev_errors_df["start"].tolist()) if len(prev_errors_df) else set()

    structural = {"malformed_item_type", "malformed_relations_type", "malformed_relation_item_type"}
    if len(prev_dropped_df) and "start" in prev_dropped_df.columns:
        mask = prev_dropped_df["reason"].astype(str).apply(lambda r: any(r.startswith(s) for s in structural))
        starts_to_retry |= set(prev_dropped_df.loc[mask, "start"].dropna().astype(int).tolist())

    print(f"Model đang dùng để resume: {GROQ_MODEL}")
    print(f"Số batch cần resume: {len(starts_to_retry)}")

    new_t, new_e, new_d = [], [], []
    for start in tqdm(sorted(starts_to_retry), desc="Resume"):
        batch = source_df.iloc[start:start + batch_size]
        t_df, e_df, d_df = run_extraction_on_slice(batch, meta, min_confidence, start_label=start)
        new_t.append(t_df); new_e.append(e_df); new_d.append(d_df)

    resumed_triples_df = pd.concat(new_t, ignore_index=True) if new_t else pd.DataFrame()
    resumed_errors_df = pd.concat(new_e, ignore_index=True) if new_e else pd.DataFrame()
    resumed_dropped_df = pd.concat(new_d, ignore_index=True) if new_d else pd.DataFrame()

    old_errors_kept = (prev_errors_df[~prev_errors_df["start"].isin(starts_to_retry)]
                        if len(prev_errors_df) else prev_errors_df)
    old_dropped_kept = (prev_dropped_df[~prev_dropped_df["start"].isin(starts_to_retry)]
                         if len(prev_dropped_df) and "start" in prev_dropped_df.columns else prev_dropped_df)

    final_triples_df = pd.concat([prev_triples_df, resumed_triples_df], ignore_index=True)
    final_errors_df = pd.concat([old_errors_kept, resumed_errors_df], ignore_index=True)
    final_dropped_df = pd.concat([old_dropped_kept, resumed_dropped_df], ignore_index=True)

    n_still_failed = (resumed_errors_df["stage"] == "api_call").sum() if len(resumed_errors_df) else 0
    print(f"Triples mới thêm: {len(resumed_triples_df):,} | Batch vẫn lỗi api_call: {n_still_failed:,}")

    return final_triples_df, final_errors_df, final_dropped_df


# ---- Kích hoạt chạy thật ----
# Nếu đã có raw_triples_df từ lần chạy trước (biến này tồn tại trong session) -> RESUME.
# Nếu chưa có -> chạy full từ đầu.
if "raw_triples_df" in globals() and raw_triples_df is not None and len(raw_triples_df) > 0:
    print("Phát hiện đã có kết quả cũ trong session -> RESUME các batch lỗi.")
    raw_triples_df, extraction_errors_df, dropped_relations_df = resume_failed_extraction(
        extraction_source, raw_triples_df, extraction_errors_df, dropped_relations_df
    )
else:
    print("Chưa có kết quả cũ -> chạy full từ đầu.")
    raw_triples_df, extraction_errors_df, dropped_relations_df = run_extraction(extraction_source)

print(f"Triples giữ lại: {len(raw_triples_df):,}")
if len(extraction_errors_df) > 0:
    print(f"Batch lỗi API call: {(extraction_errors_df['stage'] == 'api_call').sum():,}")
    print(f"Batch lỗi parse response: {(extraction_errors_df['stage'] == 'parse_response').sum():,}")
else:
    print("Batch lỗi: 0")
print(f"Relations/items bị loại bởi guardrail: {len(dropped_relations_df):,}")
if len(dropped_relations_df) > 0:
    print("Breakdown lý do loại:")
    print(dropped_relations_df["reason"].value_counts().head(15))

display(raw_triples_df.head(10))
if len(dropped_relations_df) > 0:
    display(dropped_relations_df.head(10))

Phát hiện đã có kết quả cũ trong session -> RESUME các batch lỗi.
Model đang dùng để resume: openai/gpt-oss-120b
Số batch cần resume: 0


Resume: 0it [00:00, ?it/s]

Triples mới thêm: 0 | Batch vẫn lỗi api_call: 0
Triples giữ lại: 137
Batch lỗi: 0
Relations/items bị loại bởi guardrail: 0


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence,model_used
0,Sineng Electric,Company,USES,EliteSiC,Technology,1a05beb7aa3071be6fd7::c0000,2023-05-16,Sineng Electric will integrate onsemi EliteSiC silic,0.90,openai/gpt-oss-safeguard-20b
1,GreenPages,Company,ACQUIRED,Zanaris,Company,4bd7afdba71243b0dbcd::c0000,2023-05-02,GreenPages acquired Toronto-based Zanaris,0.95,openai/gpt-oss-safeguard-20b
2,Mark Leary,Person,WORKED_AT,IDC,Company,24e2333c327a81eca0f6::c0000,2023-03-03,Mark Leary research director for network analytics and automation at IDC,1.00,openai/gpt-oss-safeguard-20b
3,dynaCERT,Company,DEVELOPED,HydraGEN™ Carbon Emission Reduction Technology,Technology,7ac7c9c97aac23e234ea::c0000,2023-09-18,dynaCERT’s HydraGEN™ Carbon Emission Reduction Technology line of commercial products,1.00,openai/gpt-oss-safeguard-20b
4,Tower Arch Capital,Company,INVESTED_IN,Intelligent Technical Solutions,Company,f982dd2920614d44fec3::c0000,2023-03-28,Intelligent Technical Solutions ('ITS') a portfolio company of Tower Arch Capital,1.00,openai/gpt-oss-safeguard-20b
5,Intelligent Technical Solutions,Company,ACQUIRED,Granite Computer Solutions,Company,f982dd2920614d44fec3::c0000,2023-03-28,Intelligent Technical Solutions ... has acquired Granite Computer Solutions ( Tempe AZ ),1.00,openai/gpt-oss-safeguard-20b
6,Intelligent Technical Solutions,Company,ACQUIRED,BrightWire Networks,Company,f982dd2920614d44fec3::c0000,2023-03-28,Intelligent Technical Solutions ... has acquired ... BrightWire Networks ( Olympia),1.00,openai/gpt-oss-safeguard-20b
7,Renovus,Company,INVESTED_IN,Aretum,Company,c93002837c287180b0c5::c0000,2023-04-19,Aretum becomes the fourth company backed by Renovus,1.00,openai/gpt-oss-safeguard-20b
8,SIOS Technology Corp.,Company,PARTNERED_WITH,ACP IT Solutions GmbH Dresden,Company,30eabd8710efff4abc2e::c0000,2023-08-14,SIOS Technology Corp. has partnered with ACP IT Solutions GmbH Dresden,1.00,openai/gpt-oss-safeguard-20b
9,Iridium Communications Inc.,Company,USES,animal tracking,Technology,f72955d8ffe5ac53791f::c0000,2022-12-14,program in support of the Smithsonian Institution''s Movement of Life Initiative which advances conservation through...,0.90,openai/gpt-oss-safeguard-20b


In [ ]:
flagged_ids = ["76879c031f77eb7392eb::c0000", "70175d5248e739d351ec::c0000"]

# 1) Có mặt trong extraction_source (đầu vào Section 2) không?
in_source = extraction_source[extraction_source["chunk_id"].isin(flagged_ids)]
print("Trong extraction_source:", len(in_source), "/", len(flagged_ids))
print(in_source[["chunk_id"]])

# 2) Có sinh ra triples nào trong raw_triples_df không? (cột là source_chunk_id, không phải chunk_id)
in_triples = raw_triples_df[raw_triples_df["source_chunk_id"].isin(flagged_ids)]
print("\nCó triples trong raw_triples_df:", len(in_triples))

# 3) Có bị guardrail loại (dropped) không?
in_dropped = dropped_relations_df[dropped_relations_df["chunk_id"].isin(flagged_ids)]
print("Có trong dropped_relations_df:", len(in_dropped))
print(in_dropped)

# 4) Batch (start) chứa 2 chunk này nằm ở đâu trong extraction_source, để tra xem batch đó
#    có từng nằm trong extraction_errors_df lịch sử không (dù giờ đã resume xong)
idx_positions = [extraction_source.index.get_loc(i) for i in in_source.index]
print("\nVị trí (index) trong extraction_source:", idx_positions)
print("Batch (start, batch_size=4) tương ứng:", [ (p // 4) * 4 for p in idx_positions ])

Trong extraction_source: 2 / 2
                        chunk_id
216  76879c031f77eb7392eb::c0000
261  70175d5248e739d351ec::c0000

Có triples trong raw_triples_df: 1
Có trong dropped_relations_df: 0
Empty DataFrame
Columns: [chunk_id, source_raw, relation, target_raw, reason]
Index: []

Vị trí (index) trong extraction_source: [216, 261]
Batch (start, batch_size=4) tương ứng: [216, 260]


In [40]:
#@title 2.2 — Entity resolution (Challenge B: type-aware merge guard)

CORP_SUFFIXES = {
    "inc", "incorporated", "corp", "corporation", "ltd", "limited", "llc",
    "plc", "co", "company", "holdings", "holding", "group", "technologies",
    "technology", "sa", "ag", "nv", "gmbh", "kk", "srl",
}

PRODUCT_DISTINGUISHING_MODIFIERS = {
    "engine", "cloud", "pro", "enterprise", "suite", "platform", "service",
    "services", "studio", "hub", "core", "edge", "lite", "plus", "max",
    "mini", "air", "prime", "advanced", "business", "for",
}

TICKER_RE = re.compile(r"^[A-Z]{1,5}$")

MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}


def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)


def is_ticker_like(raw_name):
    return bool(TICKER_RE.match(norm_space(raw_name)))


def _tokens(name):
    return set(strip_suffix(name).split())


def _guard_company(a_raw, b_raw, na, nb):
    if is_ticker_like(a_raw) or is_ticker_like(b_raw):
        return na == nb
    if na == nb:
        return True
    ratio = SequenceMatcher(None, na, nb).ratio()
    ta, tb = set(na.split()), set(nb.split())
    jaccard = len(ta & tb) / len(ta | tb) if (ta or tb) else 0.0
    return ratio >= 0.72 and jaccard > 0.0


def _guard_technology(a_raw, b_raw, na, nb):
    ta, tb = set(na.split()), set(nb.split())
    if na == nb:
        return True
    extra = (ta ^ tb)
    if extra and extra & PRODUCT_DISTINGUISHING_MODIFIERS:
        return False
    ratio = SequenceMatcher(None, na, nb).ratio()
    jaccard = len(ta & tb) / len(ta | tb) if (ta or tb) else 0.0
    return ratio >= 0.85 and jaccard >= 0.6


def _guard_person(a_raw, b_raw, na, nb):
    """
    Person guard:
    - Tách given-name / family-name.
    - Cùng họ + given-name khớp/prefix/nickname-fuzzy (ratio>=0.75) -> merge.
      (fallback fuzzy xử lý case như 'Jon' vs 'John': không phải quan hệ prefix
      ký tự thật, nhưng là biến thể tên quen thuộc -> cần fuzzy mới bắt được)
    - Cùng họ nhưng given-name khác hẳn (vd John vs Jane) -> reject.
    - Họ khác hẳn -> fallback fuzzy ratio toàn chuỗi, ngưỡng cao (0.85).
    - Mononym / lệch số token -> fuzzy ratio toàn chuỗi, ngưỡng 0.80.
    """
    ta, tb = na.split(), nb.split()
    if na == nb:
        return True

    if len(ta) >= 2 and len(tb) >= 2:
        given_a, family_a = ta[0], ta[-1]
        given_b, family_b = tb[0], tb[-1]
        if family_a == family_b:
            given_match = (
                given_a == given_b
                or given_a.startswith(given_b) or given_b.startswith(given_a)
                or SequenceMatcher(None, given_a, given_b).ratio() >= 0.75
            )
            return given_match
        ratio = SequenceMatcher(None, na, nb).ratio()
        return ratio >= 0.85

    ratio = SequenceMatcher(None, na, nb).ratio()
    return ratio >= 0.80


def merge_guard(a_raw, b_raw, typ):
    na, nb = strip_suffix(a_raw), strip_suffix(b_raw)
    if typ == "Company":
        return _guard_company(a_raw, b_raw, na, nb)
    if typ == "Technology":
        return _guard_technology(a_raw, b_raw, na, nb)
    if typ == "Person":
        return _guard_person(a_raw, b_raw, na, nb)
    return na == nb


# ---- Sanity check nhanh cho report / audit (Challenge B) ----
_GUARD_TESTS = [
    ("Company", "MSFT", "MSCI", False),
    ("Company", "MSFT", "Msft", True),
    ("Company", "Microsoft Corp", "Microsoft Corporation", True),
    ("Company", "Oracle", "Miracle", False),
    ("Technology", "Kubernetes", "Kubernetes Engine", False),
    ("Technology", "Azure", "Azure ", True),
    ("Person", "John Smith", "Jane Smith", False),
    ("Person", "Jon Smith", "John Smith", True),
    ("Person", "Elon Musk", "Elon Musk", True),
]
_failed = []
for typ, a, b, expected in _GUARD_TESTS:
    got = merge_guard(a, b, typ)
    if got != expected:
        _failed.append((typ, a, b, expected, got))
if _failed:
    print("⚠️ merge_guard sanity check FAILED cho:", _failed)
else:
    print(f"✅ merge_guard sanity check: {len(_GUARD_TESTS)}/{len(_GUARD_TESTS)} passed")


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None


def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder


class UF:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a


def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j], typ)
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)


def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()

    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n, t) for n, t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n, t) for n, t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t, n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t, n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)


# ---- Kích hoạt chạy thật ----
entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)

raw_mentions = pd.concat([
    raw_triples_df[["source_type","source_raw"]].rename(columns={"source_type":"type","source_raw":"name"}),
    raw_triples_df[["target_type","target_raw"]].rename(columns={"target_type":"type","target_raw":"name"})
])
raw_mentions["norm"] = raw_mentions["name"].map(norm_entity)
print("Số mention distinct (type, norm) TRƯỚC resolve:")
print(raw_mentions.groupby("type")["norm"].nunique())

print(f"\nSố cạnh audit (candidate pairs từ ANN, threshold=0.90): {len(entity_resolution_audit_df):,}")
if len(entity_resolution_audit_df) > 0:
    print("\nBreakdown quyết định:")
    print(entity_resolution_audit_df["decision"].value_counts())

print(f"\nTriples sau canonicalize: {len(triples_df):,} (so với raw: {len(raw_triples_df):,})")
print("\nSố entity duy nhất SAU resolve:")
print(pd.concat([
    triples_df[["source_id", "source_type"]].rename(columns={"source_id": "id", "source_type": "type"}),
    triples_df[["target_id", "target_type"]].rename(columns={"target_id": "id", "target_type": "type"})
]).drop_duplicates("id")["type"].value_counts())

display(entity_resolution_audit_df)
display(triples_df.head(10))

✅ merge_guard sanity check: 9/9 passed


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Số mention distinct (type, norm) TRƯỚC resolve:
type
Company       131
Person         21
Technology     69
Name: norm, dtype: int64

Số cạnh audit (candidate pairs từ ANN, threshold=0.90): 2

Breakdown quyết định:
decision
MERGE_VECTOR    1
REJECT_GUARD    1
Name: count, dtype: int64

Triples sau canonicalize: 137 (so với raw: 137)

Số entity duy nhất SAU resolve:
type
Company       131
Technology     68
Person         21
Name: count, dtype: int64


,type,left,right,similarity,decision
0,Technology,True Solid-State Scanning LiDAR technology,True Solid-State Scanning LiDAR,0.962421,MERGE_VECTOR
1,Technology,X90A,X90,0.920579,REJECT_GUARD


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence,model_used,source_name,target_name,source_name_norm,target_name_norm,source_id,target_id
0,Sineng Electric,Company,USES,EliteSiC,Technology,1a05beb7aa3071be6fd7::c0000,2023-05-16,Sineng Electric will integrate onsemi EliteSiC silic,0.90,openai/gpt-oss-safeguard-20b,Sineng Electric,EliteSiC,sineng electric,elitesic,b1d4bcc3cf0bdd464cb971b3,f22a43b8f0411a47aa0a37ff
1,GreenPages,Company,ACQUIRED,Zanaris,Company,4bd7afdba71243b0dbcd::c0000,2023-05-02,GreenPages acquired Toronto-based Zanaris,0.95,openai/gpt-oss-safeguard-20b,GreenPages,Zanaris,greenpages,zanaris,0430a89df77e6234ef375307,d00f157aa6483414b0dba69f
2,Mark Leary,Person,WORKED_AT,IDC,Company,24e2333c327a81eca0f6::c0000,2023-03-03,Mark Leary research director for network analytics and automation at IDC,1.00,openai/gpt-oss-safeguard-20b,Mark Leary,IDC,mark leary,idc,11fadc9a85ce38f8458803fa,5c0dfda16cbfa608d8cfc87b
3,dynaCERT,Company,DEVELOPED,HydraGEN™ Carbon Emission Reduction Technology,Technology,7ac7c9c97aac23e234ea::c0000,2023-09-18,dynaCERT’s HydraGEN™ Carbon Emission Reduction Technology line of commercial products,1.00,openai/gpt-oss-safeguard-20b,dynaCERT,HydraGEN™ Carbon Emission Reduction Technology,dynacert,hydragentm carbon emission reduction technology,93d232b3891e8ba56c2ed300,c732f180722b7251a75df55c
4,Tower Arch Capital,Company,INVESTED_IN,Intelligent Technical Solutions,Company,f982dd2920614d44fec3::c0000,2023-03-28,Intelligent Technical Solutions ('ITS') a portfolio company of Tower Arch Capital,1.00,openai/gpt-oss-safeguard-20b,Tower Arch Capital,Intelligent Technical Solutions,tower arch capital,intelligent technical solutions,80cc9e4f43076d282967e4b2,23d7ceb58c062d83135817ba
5,Intelligent Technical Solutions,Company,ACQUIRED,Granite Computer Solutions,Company,f982dd2920614d44fec3::c0000,2023-03-28,Intelligent Technical Solutions ... has acquired Granite Computer Solutions ( Tempe AZ ),1.00,openai/gpt-oss-safeguard-20b,Intelligent Technical Solutions,Granite Computer Solutions,intelligent technical solutions,granite computer solutions,23d7ceb58c062d83135817ba,71006530fc72a47a51a3f8e4
6,Intelligent Technical Solutions,Company,ACQUIRED,BrightWire Networks,Company,f982dd2920614d44fec3::c0000,2023-03-28,Intelligent Technical Solutions ... has acquired ... BrightWire Networks ( Olympia),1.00,openai/gpt-oss-safeguard-20b,Intelligent Technical Solutions,BrightWire Networks,intelligent technical solutions,brightwire networks,23d7ceb58c062d83135817ba,13560ce11a7425ac7d706b75
7,Renovus,Company,INVESTED_IN,Aretum,Company,c93002837c287180b0c5::c0000,2023-04-19,Aretum becomes the fourth company backed by Renovus,1.00,openai/gpt-oss-safeguard-20b,Renovus,Aretum,renovus,aretum,08193f3ca0efe2f71d179f4a,431cca765017665a7961a6ac
8,SIOS Technology Corp.,Company,PARTNERED_WITH,ACP IT Solutions GmbH Dresden,Company,30eabd8710efff4abc2e::c0000,2023-08-14,SIOS Technology Corp. has partnered with ACP IT Solutions GmbH Dresden,1.00,openai/gpt-oss-safeguard-20b,SIOS Technology Corp.,ACP IT Solutions GmbH Dresden,sios technology corp.,acp it solutions gmbh dresden,174ebc72eaa6debdae9049a9,c7731f8aaf2677eb3ece67e5
9,Iridium Communications Inc.,Company,USES,animal tracking,Technology,f72955d8ffe5ac53791f::c0000,2022-12-14,program in support of the Smithsonian Institution''s Movement of Life Initiative which advances conservation through...,0.90,openai/gpt-oss-safeguard-20b,Iridium Communications Inc.,animal tracking,iridium communications inc.,animal tracking,b719c247da976b81f33e935e,22929226453f9e51e80eceab


In [41]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)


In [42]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()


{'nodes': 294, 'edges': 203, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,cc9c6ee3857729e221d3f6de,ServiceNow,Company,8
1,5ebac6e7f79029d438a05ac9,Sysdig Inc.,Company,5
2,482e949a0b6c21ca2b8da03e,Infineon,Company,5
3,8ebb46d9793963ec83433906,KYY,Company,4
4,1436a8eb4a4e8197fa650e48,Opsys Tech,Company,4
5,483064d6337d5cd16bda1f38,PSG,Company,3
6,9960a1ca5dd13b4f18ca1fb7,New Charter Technologies,Company,3
7,af2cc43aae380d2715acfeff,Syndio,Company,3
8,23d7ceb58c062d83135817ba,Intelligent Technical Solutions,Company,3
9,7d87b719c7aab894bd5ffd26,Resonac,Company,3


In [43]:
#@title 3.1 — Flat RAG baseline
# Dùng cùng embedding/generator với GraphRAG để comparison tập trung vào
# retrieval architecture (flat vector search vs graph traversal), không lẫn
# yếu tố khác biệt về model.

flat_index = None
flat_store = None


def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")
    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print(f"✅ Flat vectors indexed: {flat_index.ntotal:,}")


def retrieve_flat_context(query, k=6):
    if flat_index is None:
        raise RuntimeError("Hãy chạy build_flat_index(chunks_df) trước.")

    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score": float(score), "chunk_id": r.chunk_id,
            "published_date": r.published_date, "text": r.text
        })
    df = pd.DataFrame(rows)

    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df


# ---- Kích hoạt chạy thật ----
build_flat_index(chunks_df)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Flat vectors indexed: 1,496


In [44]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)


In [45]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]


In [46]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out


In [47]:
#@title 4.1 — Load Golden Dataset (thật, 50 câu, đã có sẵn reference_answer)

GOLDEN_PATH = "/content/graphrag_golden_50_first5000.csv"            # 5 cột chuẩn, dùng cho eval chính
GOLDEN_DETAILED_PATH = "/content/graphrag_golden_50_first5000_detailed.csv"  # có thêm scoring_notes, gold_reasoning... dùng cho judge

def validate_golden(df, require_answers=True):
    required = {"id", "group", "question", "reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required - set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id", "question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    if df["id"].duplicated().any():
        display(df[df["id"].duplicated(keep=False)])
        raise ValueError("Có id bị trùng trong golden dataset.")
    print(f"✅ Golden Dataset valid: {len(df)} câu, nhóm: {dict(df['group'].value_counts())}")

golden_df = pd.read_csv(GOLDEN_PATH)
golden_detailed_df = pd.read_csv(GOLDEN_DETAILED_PATH)  # bản đầy đủ, dùng ở Section 4 khi chấm điểm

validate_golden(golden_df)
display(golden_df.head())

✅ Golden Dataset valid: 50 câu, nhóm: {'multi-hop': np.int64(23), 'cross-doc': np.int64(22), 'factoid': np.int64(5)}


,id,group,question,reference_answer,reference_evidence
0,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
1,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
2,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.",row 935 (2023-01-18 22:37:00): A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular...
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
4,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 935 (2023-01-18 22:37:00): A Leap in...


In [88]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out


In [93]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = "/content/graphrag_eval_checkpoint.csv"
ERROR_LOG = "/content/graphrag_eval_errors.csv"

N_EVAL_SAMPLES = 10  # <-- đổi số câu muốn chạy ở đây (10, 11, hoặc None để chạy full 50)

def get_eval_subset(golden_df, n=None, stratified=True):
    """
    Lấy subset để eval, tiết kiệm quota. Nếu stratified=True, lấy đều theo từng
    group (multi-hop/cross-doc/factoid) theo tỉ lệ gần đúng thay vì chỉ lấy
    n câu đầu tiên (tránh thiên lệch, vd n=10 mà toàn rơi vào multi-hop).
    """
    if n is None or n >= len(golden_df):
        return golden_df.reset_index(drop=True)

    if not stratified:
        return golden_df.head(n).reset_index(drop=True)

    frac = n / len(golden_df)
    sampled = (
        golden_df.groupby("group", group_keys=False)
        .apply(lambda g: g.sample(max(1, round(len(g) * frac)), random_state=SEED))
    )
    sampled = sampled.head(n).reset_index(drop=True)  # đảm bảo không vượt quá n do làm tròn
    return sampled


def run_evaluation(golden_df):
    if Path(CHECKPOINT).exists():
        prev_df = pd.read_csv(CHECKPOINT)
        done_ids = set(prev_df["id"].tolist())
        print(f"Phát hiện checkpoint cũ: {len(prev_df)} câu đã xong -> resume phần còn lại.")
    else:
        prev_df = pd.DataFrame()
        done_ids = set()

    remaining = golden_df[~golden_df["id"].isin(done_ids)]
    print(f"Số câu cần chạy: {len(remaining)}/{len(golden_df)}")

    rows = prev_df.to_dict("records")
    errors = []

    for q in tqdm(remaining.itertuples(index=False), total=len(remaining), desc="Evaluation"):
        try:
            flat = answer_flat_rag(q.question)
            graph = answer_graph_rag(q.question)
            jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
            jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])
        except Exception as e:
            errors.append({"id": q.id, "question": q.question, "error": str(e)})
            pd.DataFrame(errors).to_csv(ERROR_LOG, index=False)
            continue

        rows.append({
            "id": q.id, "group": q.group, "question": q.question,
            "reference_answer": q.reference_answer,
            "flat_answer": flat["answer"], "graph_answer": graph["answer"],
            "flat_comprehensiveness": jf["comprehensiveness"],
            "graph_comprehensiveness": jg["comprehensiveness"],
            "flat_faithfulness": jf["faithfulness"],
            "graph_faithfulness": jg["faithfulness"],
            "flat_multi_hop_reasoning": jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning": jg["multi_hop_reasoning"],
            "flat_latency_s": flat["latency_s"],
            "graph_latency_s": graph["latency_s"],
            "flat_total_tokens": flat.get("total_tokens"),
            "graph_total_tokens": graph.get("total_tokens"),
            "flat_judge_rationale": jf["rationale"],
            "graph_judge_rationale": jg["rationale"],
            "graph_supernode_events": len(
                graph["graph_debug"]["diagnostics"].get("supernode_events", [])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

    if errors:
        print(f"⚠️ {len(errors)} câu bị lỗi (xem {ERROR_LOG}), chưa nằm trong checkpoint -> chạy lại cell này để resume.")
    return pd.DataFrame(rows)


eval_subset_df = get_eval_subset(golden_df, n=N_EVAL_SAMPLES, stratified=True)
validate_golden(eval_subset_df, require_answers=True)
eval_results_df = run_evaluation(eval_subset_df)
display(eval_results_df)

✅ Golden Dataset valid: 10 câu, nhóm: {'multi-hop': np.int64(5), 'cross-doc': np.int64(4), 'factoid': np.int64(1)}
Phát hiện checkpoint cũ: 10 câu đã xong -> resume phần còn lại.
Số câu cần chạy: 0/10


/tmp/ipykernel_1948/2541935772.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(max(1, round(len(g) * frac)), random_state=SEED))


Evaluation: 0it [00:00, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,"I’m sorry, but none of the provided chunks mention Aeris, Ericsson, or any related acquisition or transfer. Therefor...","I’m sorry, but the supplied context does not contain any information about Aeris/Ericsson reports, so I can’t determ...",1,1,1,1,1,1,0.347397,0.499727,787,661,"The candidate answer incorrectly states that no relevant information is present, whereas the reference indicates tha...","The candidate answer fails to address the question, incorrectly stating that no information is available, whereas th...",0
1,G5000-32,cross-doc,What is the difference between OpenAI's March plug-in development and its June reported app-store plan?,The March story describes ChatGPT gaining support for application plug-ins so companies can expose product functiona...,"I’m sorry, but the supplied context does not contain information about OpenAI’s March plug‑in development or a June‑...","I’m sorry, but the supplied context does not contain any information about OpenAI’s plug‑in development in March or ...",1,1,1,1,1,1,0.391072,0.467719,765,891,The candidate answer does not provide any comparison between the March plug‑in development and the June app‑store pl...,The candidate answer incorrectly claims that the supplied context contains no information about OpenAI’s March plug‑...,0
2,G5000-20,cross-doc,How did Options Technology's Microsoft partner designations expand from May to June 2023?,"In May, Options Technology announced Microsoft Solutions Partner status for Infrastructure. In June, it announced Mi...",In May 2023 Options Technology announced that it had earned the **Microsoft Solutions Partner designation for Infras...,The only information provided about Options Technology’s Microsoft partner status is a May 2023 announcement that th...,1,1,1,1,1,1,0.689239,0.921590,828,881,"The candidate only mentions the May Infrastructure designation and incorrectly claims no June information, missing t...","The reference clearly states that Options Technology added a Security designation in June, expanding beyond the May ...",0
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,"The supplied excerpts do not contain any references to Ericsson, its IoT businesses, or the Aeris transaction. Conse...","I’m sorry, but the supplied excerpts do not contain any references to Ericsson’s IoT businesses or to the Aeris tran...",1,1,1,1,1,1,0.519426,0.607860,778,653,The candidate answer fails to mention the two named Ericsson IoT businesses (IoT Accelerator and Connected Vehicle C...,The candidate answer fails to mention the two named Ericsson IoT businesses (IoT Accelerator and Connected Vehicle C...,0
4,G5000-13,factoid,"Which three companies launched AI Lighthouse in the selected 5,000-row scope?","ServiceNow, NVIDIA, and Accenture.","The AI Lighthouse program was launched jointly by **ServiceNow, NVIDIA, and Accenture**【chunk_id=a5a8b0ece135c638d4f...","The AI Lighthouse program was launched jointly by **ServiceNow**, **NVIDIA**, and **Accenture**【chunk_id=a5a8b0ece13...",5,5,5,5,5,5,0.555065,0.519860,697,796,"The candidate correctly lists all three companies—ServiceNow, NVIDIA, and Accenture—matching the reference answer ex...","The candidate lists all t

In [73]:
print("Key hiện tại (10 ký tự cuối):", GROQ_API_KEY[-10:] if GROQ_API_KEY else "TRỐNG")
print("Model hiện tại:", GROQ_MODEL, "| Judge model:", JUDGE_MODEL)

Key hiện tại (10 ký tự cuối): B9GbQjaBgG
Model hiện tại: openai/gpt-oss-120b | Judge model: openai/gpt-oss-20b


In [92]:
errors_df = pd.read_csv("/content/graphrag_eval_errors.csv")
print(len(errors_df))
print(errors_df["error"].iloc[0])  # xem 1 lỗi mẫu đầy đủ
print(errors_df["error"].value_counts().head(5))  # xem có bao nhiêu loại lỗi khác nhau

8
RATE_LIMIT_LIKELY_EXHAUSTED: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kc0wewezfh1tx09rnp5zdftp` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 200000, Requested 821. Please try again in 5m54.672s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
error
RATE_LIMIT_LIKELY_EXHAUSTED: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kc0wewezfh1tx09rnp5zdftp` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 200000, Requested 821. Please try again in 5m54.672s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}    1
RATE_LIMIT_LIKELY_EXHAUSTED: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b

In [95]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv("/content/graphrag_eval_results.csv", index=False)
comparison_df.to_csv("/content/graphrag_vs_flatrag_summary.csv", index=False)


,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,1.000,1.000,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,1.000,1.000,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,1.000,1.000,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),0.487,0.624,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,789.500,771.500,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,5.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),0.555,0.520,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,697.000,796.000,Flat RAG thường rẻ/nhanh hơn.


In [96]:
#@title 5.1 — Bắt buộc chứng minh (provenance, resolution audit, super-node cap, comparison table)

# ---- 1. Edge provenance không thiếu ----
def test_edge_provenance():
    total = run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]
    missing = run_cypher("""
        MATCH ()-[r]->()
        WHERE r.source_chunk_id IS NULL OR r.source_chunk_id = ''
           OR r.published_date IS NULL OR r.published_date = ''
        RETURN count(r) AS c
    """)[0]["c"]
    print(f"Tổng edges trong graph: {total:,} | Thiếu provenance: {missing:,}")
    assert missing == 0, f"❌ {missing} edges thiếu source_chunk_id/published_date"
    print("✅ Edge provenance đầy đủ (100% edges có source_chunk_id + published_date).")


# ---- 2. Entity Resolution có audit ----
def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("⚠️ No audit rows — kiểm tra lại build_resolution_map() đã chạy chưa.")
        return
    print(f"Tổng candidate pairs được audit: {len(audit_df):,}")
    print(audit_df["decision"].value_counts())
    print("\nTop similarity pairs:")
    display(audit_df.sort_values("similarity", ascending=False).head(30))
    print("\nHigh-similarity nhưng bị guard reject (bằng chứng guard có tác dụng thật):")
    display(
        audit_df[audit_df.decision == "REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )
    print("✅ Entity Resolution audit đầy đủ.")


# ---- 3. Super-node degree > 100 chỉ expand tối đa 50 edge ----
def test_supernode_policy():
    rows = run_cypher("""
        MATCH (n:Entity)-[r]-()
        WITH n, count(r) AS degree
        ORDER BY degree DESC LIMIT 1
        RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("⚠️ Graph empty.")
        return
    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "| fetched =", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50, f"❌ Fetch {len(edges)} edges, vượt cap 50."
        print("✅ Super-node cap OK (degree > 100 -> chỉ lấy tối đa 50 edge).")
    else:
        print(f"ℹ️ Node top-degree ({n['degree']}) chưa vượt SUPER_NODE_DEGREE={SUPER_NODE_DEGREE}, chưa trigger cap.")


# ---- 4. Comparison table (đã export ở 4.4, chỉ hiển thị lại để đủ bộ chứng minh) ----
def show_comparison_summary(eval_results_df):
    print(f"Comparison table: {len(eval_results_df)} câu đã eval.")
    summary = eval_results_df[[
        "flat_comprehensiveness", "graph_comprehensiveness",
        "flat_faithfulness", "graph_faithfulness",
        "flat_multi_hop_reasoning", "graph_multi_hop_reasoning",
        "flat_latency_s", "graph_latency_s",
    ]].mean(numeric_only=True)
    display(summary.to_frame("mean"))
    print("✅ Comparison table sẵn sàng (đã export ở 4.4).")


# ---- Chạy toàn bộ 4 chứng minh ----
print("=" * 60, "\n1. EDGE PROVENANCE\n", "=" * 60)
test_edge_provenance()

print("\n", "=" * 60, "\n2. ENTITY RESOLUTION AUDIT\n", "=" * 60)
show_resolution_audit(entity_resolution_audit_df)

print("\n", "=" * 60, "\n3. SUPER-NODE CAP\n", "=" * 60)
test_supernode_policy()

print("\n", "=" * 60, "\n4. COMPARISON TABLE\n", "=" * 60)
show_comparison_summary(eval_results_df)

1. EDGE PROVENANCE
Tổng edges trong graph: 203 | Thiếu provenance: 0
✅ Edge provenance đầy đủ (100% edges có source_chunk_id + published_date).

2. ENTITY RESOLUTION AUDIT
Tổng candidate pairs được audit: 2
decision
MERGE_VECTOR    1
REJECT_GUARD    1
Name: count, dtype: int64

Top similarity pairs:


,type,left,right,similarity,decision
0,Technology,True Solid-State Scanning LiDAR technology,True Solid-State Scanning LiDAR,0.962421,MERGE_VECTOR
1,Technology,X90A,X90,0.920579,REJECT_GUARD



High-similarity nhưng bị guard reject (bằng chứng guard có tác dụng thật):


,type,left,right,similarity,decision
1,Technology,X90A,X90,0.920579,REJECT_GUARD


✅ Entity Resolution audit đầy đủ.

3. SUPER-NODE CAP
{'id': 'cc9c6ee3857729e221d3f6de', 'name': 'ServiceNow', 'degree': 8} | fetched = 8
ℹ️ Node top-degree (8) chưa vượt SUPER_NODE_DEGREE=100, chưa trigger cap.

4. COMPARISON TABLE
Comparison table: 10 câu đã eval.


,mean
flat_comprehensiveness,2.200000
graph_comprehensiveness,2.200000
flat_faithfulness,2.600000
graph_faithfulness,2.200000
flat_multi_hop_reasoning,2.200000
graph_multi_hop_reasoning,2.200000
flat_latency_s,2.835855
graph_latency_s,2.717769


✅ Comparison table sẵn sàng (đã export ở 4.4).
